# Loan Data Processing
Preprocessing pipeline for loan approval dataset.

## Step 1 — Load & Inspect

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

df = pd.read_csv("../../../dataset/raw_loan_dataset.csv")
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved
0,108810,537.0,1.1,25800,Yes,No,No
1,96482,524.0,15.0,11200,Y,No,Yes
2,28478,NaN,5.4,12100,No,No,Yes
3,"$25,851",561.0,17.6,7000,Yes,No,Yes
4,38396,527.0,9.8,10700,No,No,Approved


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 103 entries, 0 to 102
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Income            98 non-null     object 
 1   CreditScore       97 non-null     float64
 2   EmploymentYears   98 non-null     float64
 3   LoanAmount        98 non-null     object 
 4   HasCollateral     101 non-null    object 
 5   PreviousDefaults  101 non-null    object 
 6   Approved          103 non-null    object 
dtypes: float64(2), object(5)
memory usage: 5.8+ KB


In [3]:
df.isnull().sum()

Income              5
CreditScore         6
EmploymentYears     5
LoanAmount          5
HasCollateral       2
PreviousDefaults    2
Approved            0
dtype: int64

In [4]:
# Key issues:
# - Income and LoanAmount have $ signs and commas
# - HasCollateral, PreviousDefaults, Approved have typos and mixed casing
# - CreditScore, EmploymentYears, Income, LoanAmount have missing values
df['HasCollateral'].value_counts(dropna=False)

HasCollateral
No     50
Yes    46
Y       2
NaN     2
N       1
yse     1
yes     1
Name: count, dtype: int64

In [5]:
df['PreviousDefaults'].value_counts(dropna=False)

PreviousDefaults
No     84
Yes    13
NaN     2
1       1
0       1
Y       1
N       1
Name: count, dtype: int64

In [6]:
df['Approved'].value_counts(dropna=False)

Approved
Yes         65
No          32
Approved     1
Rejected     1
approved     1
rejected     1
YES          1
NO           1
Name: count, dtype: int64

## Step 2 — Clean Currency Formatting

In [7]:
df['Income'] = df['Income'].replace(r'[\$,]', '', regex=True).astype(float)
df['LoanAmount'] = df['LoanAmount'].replace(r'[\$,]', '', regex=True).astype(float)
df[['Income', 'LoanAmount']].dtypes

Income        float64
LoanAmount    float64
dtype: object

## Step 3 — Fix Category Errors Before Imputation

In [8]:
yes_no_map = {
    'yes': 'Yes', 'y': 'Yes', 'yse': 'Yes', '1': 'Yes', 'approved': 'Yes',
    'no': 'No',  'n': 'No',  '0': 'No',  'rejected': 'No',
}

for col in ['HasCollateral', 'PreviousDefaults', 'Approved']:
    df[col] = df[col].astype(str).str.strip().str.lower().replace(yes_no_map).replace({'nan': np.nan})

df['HasCollateral'].value_counts(dropna=False)

HasCollateral
No     51
Yes    50
NaN     2
Name: count, dtype: int64

In [9]:
df['PreviousDefaults'].value_counts(dropna=False)

PreviousDefaults
No     86
Yes    15
NaN     2
Name: count, dtype: int64

In [10]:
df['Approved'].value_counts(dropna=False)

Approved
Yes    68
No     35
Name: count, dtype: int64

## Step 4 — Impute Missing Values

In [11]:
df['Income']          = df['Income'].fillna(df['Income'].median())
df['CreditScore']     = df['CreditScore'].fillna(df['CreditScore'].median())
df['EmploymentYears'] = df['EmploymentYears'].fillna(df['EmploymentYears'].median())
df['LoanAmount']      = df['LoanAmount'].fillna(df['LoanAmount'].median())
df['HasCollateral']   = df['HasCollateral'].fillna(df['HasCollateral'].mode()[0])
df['PreviousDefaults']= df['PreviousDefaults'].fillna(df['PreviousDefaults'].mode()[0])
df.isnull().sum()

Income              0
CreditScore         0
EmploymentYears     0
LoanAmount          0
HasCollateral       0
PreviousDefaults    0
Approved            0
dtype: int64

## Step 5 — Remove Duplicates

In [12]:
before = df.shape[0]
df = df.drop_duplicates()
f"Rows before: {before} — Rows after: {df.shape[0]} — Removed: {before - df.shape[0]}" 

'Rows before: 103 — Rows after: 100 — Removed: 3'

## Step 6 — IQR Capping

In [13]:
def iqr_fun(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

for col in ['Income', 'CreditScore', 'LoanAmount', 'EmploymentYears']:
    low, high = iqr_fun(df[col])
    df[col] = df[col].clip(lower=low, upper=high)

df[['Income', 'CreditScore', 'LoanAmount', 'EmploymentYears']].describe().round(2)

,Income,CreditScore,LoanAmount,EmploymentYears
count,100.00,100.00,100.00,100.00
mean,72220.23,651.10,25507.12,12.65
std,29179.39,96.22,14793.84,7.01
min,25851.00,484.00,5000.00,0.50
25%,47964.75,576.00,14400.00,6.28
50%,69460.50,638.00,20700.00,12.55
75%,95826.50,730.50,35125.00,17.98
max,167619.12,920.00,66212.50,25.00


## Step 7 — Label Encoding

In [14]:
df['Approved']         = df['Approved'].map({'Yes': 1, 'No': 0}).astype(int)
df['HasCollateral']    = df['HasCollateral'].map({'Yes': 1, 'No': 0}).astype(int)
df['PreviousDefaults'] = df['PreviousDefaults'].map({'Yes': 1, 'No': 0}).astype(int)
df['Approved'].value_counts()

Approved
1    66
0    34
Name: count, dtype: int64

## Step 8 — Class Balance Check

In [15]:
df['Approved'].value_counts(normalize=True).round(3)
# If one class is below 30% the dataset is imbalanced
# and Accuracy alone becomes a misleading metric

Approved
1    0.66
0    0.34
Name: proportion, dtype: float64

## Step 9 — Feature Engineering

In [16]:
df['DebtToIncome']          = df['LoanAmount'] / df['Income'].replace(0, np.nan)
df['IncomePerYearEmployed'] = df['Income'] / (df['EmploymentYears'] + 1)
df[['DebtToIncome', 'IncomePerYearEmployed']].head()

,DebtToIncome,IncomePerYearEmployed
0,0.237111,51814.285714
1,0.116084,6030.125000
2,0.424889,4449.687500
3,0.270783,1389.838710
4,0.278675,3555.185185


## Step 10 — Feature Scaling (RobustScaler)

I chose **RobustScaler** instead of StandardScaler because it scales using the median and IQR rather than the mean and standard deviation. This makes it less sensitive to outliers — even after IQR capping, loan data like Income and LoanAmount can still have skewed distributions where RobustScaler is more stable.

**Source:** scikit-learn documentation — [RobustScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)

In [17]:
binary_cols = {'HasCollateral', 'PreviousDefaults', 'Approved'}
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
scale_cols = [c for c in numeric_cols if c not in binary_cols]

df[scale_cols] = RobustScaler().fit_transform(df[scale_cols])
df[scale_cols].head()

,Income,CreditScore,EmploymentYears,LoanAmount,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,-0.445244,8.274536
1,0.564574,-0.737864,0.209402,-0.458384,-0.912749,0.153994
2,-0.856268,0.000000,-0.611111,-0.414958,0.280113,-0.126321
3,-0.911156,-0.498382,0.431624,-0.661037,-0.315175,-0.669033
4,-0.649046,-0.718447,-0.235043,-0.482509,-0.284688,-0.284975


## Step 11 — Final Checks & Save

In [18]:
df.isnull().sum()

Income                   0
CreditScore              0
EmploymentYears          0
LoanAmount               0
HasCollateral            0
PreviousDefaults         0
Approved                 0
DebtToIncome             0
IncomePerYearEmployed    0
dtype: int64

In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100 entries, 0 to 99
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Income                 100 non-null    float64
 1   CreditScore            100 non-null    float64
 2   EmploymentYears        100 non-null    float64
 3   LoanAmount             100 non-null    float64
 4   HasCollateral          100 non-null    int64  
 5   PreviousDefaults       100 non-null    int64  
 6   Approved               100 non-null    int64  
 7   DebtToIncome           100 non-null    float64
 8   IncomePerYearEmployed  100 non-null    float64
dtypes: float64(6), int64(3)
memory usage: 7.8 KB


In [20]:
df.head()

,Income,CreditScore,EmploymentYears,LoanAmount,HasCollateral,PreviousDefaults,Approved,DebtToIncome,IncomePerYearEmployed
0,0.822149,-0.653722,-0.978632,0.246080,1,0,0,-0.445244,8.274536
1,0.564574,-0.737864,0.209402,-0.458384,1,0,1,-0.912749,0.153994
2,-0.856268,0.000000,-0.611111,-0.414958,0,0,1,0.280113,-0.126321
3,-0.911156,-0.498382,0.431624,-0.661037,1,0,1,-0.315175,-0.669033
4,-0.649046,-0.718447,-0.235043,-0.482509,0,0,1,-0.284688,-0.284975


In [ ]:
df.to_csv("clean_loan_dataset.csv", index=False)
print("Saved.")